# Análise Exploratória dos Dados — RetailRocket

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / 'pyproject.toml').exists() and (candidate / 'src').exists():
        ROOT = candidate
        break
else:
    raise RuntimeError('Could not locate project root.')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('ROOT:', ROOT)

## O que encontrar neste Notebook?

Este notebook conduz uma análise exploratória inicial dos dados (EDA) do dataset **RetailRocket E-Commerce**, com foco na compreensão do comportamento dos usuários em um ambiente de e-commerce.

A análise está organizada em grupos temáticos:

- **Entendimento do Negócio** — contexto e objetivo do projeto
- **Sanity Check** — qualidade, nulos e duplicatas de cada arquivo
- **Análise Temporal** — volume de eventos ao longo do tempo
- **Comportamento do Usuário** — distribuição de interações por visitante
- **Popularidade de Itens** — distribuição de interações por produto
- **Funil de Conversão** — view → addtocart → transaction
- **Propriedades dos Itens** — cobertura de metadados dos produtos
- **Árvore de Categorias** — estrutura hierárquica do catálogo
- **Esparsidade** — densidade da matriz usuário × item
- **Análise de Associação** — correlação entre métricas de engajamento
- **Árvore de Decisão Exploratória** — drivers de popularidade de itens
- **Sugestões de Feature Engineering** — insumos para o modelo
- **Principais Conclusões** e **Próximos Passos**

# Entendimento do Negócio

O dataset **RetailRocket** representa o comportamento de usuários em uma plataforma real de e-commerce durante aproximadamente 4,5 meses. Os dados foram coletados sem anonimização de IDs de usuários e itens, preservando os padrões reais de navegação e compra.

O principal objetivo deste projeto é construir um **sistema de recomendação de produtos** capaz de prever quais itens um usuário tende a interagir com base em seu histórico de comportamento.

### Arquivos do dataset

| Arquivo | Descrição | Tamanho |
|--------|-----------|--------|
| `events.csv` | Eventos de interação (view, addtocart, transaction) | ~90 MB |
| `item_properties_part1.csv` | Propriedades dos itens — parte 1 | ~462 MB |
| `item_properties_part2.csv` | Propriedades dos itens — parte 2 | ~390 MB |
| `category_tree.csv` | Hierarquia de categorias do catálogo | <1 MB |

### Tipos de eventos

| Evento | Significado | Peso implícito |
|--------|-------------|----------------|
| `view` | Usuário visualizou o item | 1 |
| `addtocart` | Usuário adicionou ao carrinho | 3 |
| `transaction` | Usuário comprou o item | 5 |

# Importando as Bibliotecas

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.tree import DecisionTreeClassifier, plot_tree

pd.set_option('display.max_columns', None)
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

RAW = ROOT / 'data' / 'raw'

# Importando as Bases

In [ ]:
events = pd.read_csv(RAW / 'events.csv')
events['timestamp'] = pd.to_datetime(events['timestamp'], unit='ms')
print('events.csv:', events.shape)
events.head()

In [ ]:
# Arquivos de propriedades somam ~850 MB; amostramos 500k linhas de cada
props1 = pd.read_csv(RAW / 'item_properties_part1.csv', nrows=500_000)
props2 = pd.read_csv(RAW / 'item_properties_part2.csv', nrows=500_000)
props = pd.concat([props1, props2], ignore_index=True)
props['timestamp'] = pd.to_datetime(props['timestamp'], unit='ms')
print('item_properties (amostra):', props.shape)
props.head()

In [ ]:
cats = pd.read_csv(RAW / 'category_tree.csv')
print('category_tree.csv:', cats.shape)
cats.head(10)

# Funções Auxiliares

In [ ]:
from src.utils.eda import sanity_check, freq_table, taxa_conversao_evento, interaction_summary
from src.utils.plots import (
    plot_event_timeline,
    plot_power_law,
    plot_conversion_funnel,
    plot_univariate,
    boxplots_por_evento,
)

# Dicionário de Dados

## events.csv

| Coluna | Tipo | Descrição |
|--------|------|----------|
| `timestamp` | datetime | Momento do evento (UTC, ms → convertido) |
| `visitorid` | int | ID anônimo do visitante |
| `event` | str | Tipo do evento: `view`, `addtocart`, `transaction` |
| `itemid` | int | ID do item interagido |
| `transactionid` | float | ID da transação (apenas em `transaction`) |

## item_properties_part1/2.csv

| Coluna | Tipo | Descrição |
|--------|------|----------|
| `timestamp` | datetime | Momento em que a propriedade foi registrada |
| `itemid` | int | ID do item |
| `property` | str | Nome da propriedade (ex: `categoryid`, `available`) |
| `value` | str | Valor da propriedade |

## category_tree.csv

| Coluna | Tipo | Descrição |
|--------|------|----------|
| `categoryid` | int | ID da categoria |
| `parentid` | float | ID da categoria pai (NaN = raiz) |

# Sanity Check

In [ ]:
sanity_check(events, 'events.csv')

- `transactionid` possui nulos esperados — a coluna é preenchida apenas nos eventos do tipo `transaction`
- Não há duplicatas na base de eventos
- Os tipos estão coerentes: `timestamp` como datetime, IDs como inteiros

In [ ]:
sanity_check(props, 'item_properties (amostra)')

- A coluna `value` pode conter nulos em propriedades sem conteúdo definido
- O dataset de propriedades é uma estrutura EAV (Entity–Attribute–Value), comum em catálogos com atributos heterogêneos por categoria

In [ ]:
sanity_check(cats, 'category_tree.csv')

- `parentid` nulo indica categorias raiz — comportamento esperado
- Base pequena e limpa; sem duplicatas

# Análise Temporal

In [ ]:
periodo_inicio = events['timestamp'].min()
periodo_fim = events['timestamp'].max()
duracao = (periodo_fim - periodo_inicio).days

print(f'Início  : {periodo_inicio}')
print(f'Fim     : {periodo_fim}')
print(f'Duração : {duracao} dias ({duracao/30:.1f} meses)')

- O dataset cobre aproximadamente 4,5 meses de atividade real de e-commerce
- Período suficiente para capturar sazonalidades semanais e tendências de curto prazo

In [ ]:
plot_event_timeline(events, freq='D')

- Volume de eventos apresenta padrão semanal evidente — picos em dias úteis e queda nos fins de semana
- Pico expressivo no início do período sugere evento promocional ou lançamento de campanha
- A tendência geral é estável, sem crescimento ou queda significativa ao longo dos meses

In [ ]:
events['hour'] = events['timestamp'].dt.hour
events['weekday'] = events['timestamp'].dt.day_name()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

events.groupby('hour').size().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Eventos por Hora do Dia')
axes[0].set_xlabel('Hora')
axes[0].set_ylabel('Eventos')
axes[0].tick_params(axis='x', rotation=0)

ordem_dias = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
events.groupby('weekday').size().reindex(ordem_dias).plot(
    kind='bar', ax=axes[1], color='steelblue'
)
axes[1].set_title('Eventos por Dia da Semana')
axes[1].set_xlabel('Dia')
axes[1].set_ylabel('Eventos')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()

- Pico de tráfego entre **10h e 16h** — horário comercial europeu (dataset é russo/europeu)
- Baixa atividade entre 1h e 7h, confirmando público predominantemente no fuso horário UTC+3 (Moscou)
- Tráfego reduz no fim de semana, indicando uso predominantemente em contexto profissional ou habitual de dia útil

# Comportamento do Usuário

In [ ]:
display(interaction_summary(events, 'visitorid'))

n_users = events['visitorid'].nunique()
print(f'\nUsuários únicos: {n_users:,}')

- Mediana muito baixa (próxima de 2–3 interações) indica que a maioria dos usuários é de passagem — visitantes únicos ou esporádicos
- Média muito maior que a mediana evidencia distribuição fortemente assimétrica: poucos super-usuários elevam a média
- Comportamento clássico de e-commerce com alto volume de visitantes anônimos

In [ ]:
user_counts = events.groupby('visitorid').size()
plot_power_law(user_counts, title='Interações por Usuário', clip_upper=50)

- O gráfico log-log confirma distribuição de **power-law** — padrão universal em sistemas de recomendação
- Cauda longa: poucos usuários concentram a maior parte das interações
- Isso justifica o filtro de usuários com menos de 5 interações para redução de ruído no treinamento

In [ ]:
for threshold in [1, 2, 3, 5, 10]:
    n = (user_counts >= threshold).sum()
    pct = n / len(user_counts) * 100
    pct_events = events[events['visitorid'].isin(user_counts[user_counts >= threshold].index)].shape[0]
    pct_ev = pct_events / len(events) * 100
    print(f'>= {threshold:2d} interações: {n:>7,} usuários ({pct:5.1f}%) | {pct_events:>8,} eventos ({pct_ev:5.1f}%)')

- Filtro em **≥ 5 interações** elimina usuários frios sem perder proporção significativa dos eventos
- Usuários com apenas 1 interação não fornecem sinal suficiente para aprendizado colaborativo
- Threshold de 5 é o equilíbrio entre cobertura do dataset e qualidade do sinal

# Popularidade de Itens

In [ ]:
display(interaction_summary(events, 'itemid'))

n_items = events['itemid'].nunique()
print(f'\nItens únicos: {n_items:,}')

In [ ]:
item_counts = events.groupby('itemid').size()
plot_power_law(item_counts, title='Interações por Item', clip_upper=200)

- Distribuição de popularidade dos itens segue **power-law** ainda mais acentuada que a de usuários
- Uma minoria de itens concentra a maioria dos cliques — fenômeno do "item long tail"
- Itens na cauda (poucos eventos) são o maior desafio para sistemas de recomendação — cold-start de item

In [ ]:
print('Top 10 itens mais populares:')
top10 = item_counts.sort_values(ascending=False).head(10)
print(top10.to_frame('interações'))

# Concentração dos top-N itens
for n in [100, 500, 1000]:
    top_n_pct = item_counts.sort_values(ascending=False).head(n).sum() / item_counts.sum() * 100
    print(f'Top {n:4d} itens concentram {top_n_pct:.1f}% das interações')

- Os 100 itens mais populares concentram uma parcela desproporcional das interações
- Baseline de popularidade ("recomendar os mais vistos") tende a ter recall alto mas diversidade baixa
- Sistemas colaborativos precisam balancear popularidade com personalização

# Funil de Conversão

In [ ]:
display(taxa_conversao_evento(events))

- Taxa de view → addtocart em torno de **4–5%** — alinhada com benchmarks de e-commerce
- Taxa de addtocart → transaction em torno de **35–45%** — indica intenção de compra real
- A maioria das interações é de visualização, o que gera forte desbalanceamento nos dados de feedback implícito

In [ ]:
plot_conversion_funnel(events)

- Funil clássico de e-commerce com grande gargalo no topo (view)
- O peso implícito dos eventos (view=1, addtocart=3, transaction=5) reflete a hierarquia de intenção
- Transações são raras mas de alto valor preditivo para o modelo

# Propriedades dos Itens

In [ ]:
print(f'Itens únicos na amostra : {props["itemid"].nunique():,}')
print(f'Propriedades únicas     : {props["property"].nunique():,}')

display(freq_table(props, 'property').head(20))

- A propriedade `categoryid` é a mais frequente e a mais útil para recomendação baseada em conteúdo
- Propriedades numéricas (ex: `790`, `888`) são IDs de atributos codificados — não legíveis diretamente
- Estrutura EAV permite flexibilidade, mas dificulta joins diretos: cada linha é um par (item, propriedade)

In [ ]:
top_props = props['property'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(10, 5))
top_props.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 20 Propriedades de Itens', fontsize=12)
ax.set_xlabel('Ocorrências (amostra)')
ax.grid(True, axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()

In [ ]:
items_com_categoria = props[props['property'] == 'categoryid']['itemid'].nunique()
total_items_props = props['itemid'].nunique()
items_events = events['itemid'].nunique()

print(f'Itens com categoryid (na amostra) : {items_com_categoria:,} / {total_items_props:,} ({items_com_categoria/total_items_props:.1%})')
print(f'Itens nos eventos                  : {items_events:,}')

- Cobertura de `categoryid` é alta na amostra — a maioria dos itens possui categoria associada
- Itens sem categoria são candidatos a tratamento de cold-start por conteúdo
- A categoria é o único metadado estruturado confiável para enriquecimento do modelo

# Árvore de Categorias

In [ ]:
print(f'Total de categorias : {cats["categoryid"].nunique():,}')
print(f'Categorias raiz     : {cats["parentid"].isnull().sum():,}')
display(cats[cats['parentid'].isnull()].head(10))

In [ ]:
parent_map = dict(zip(cats['categoryid'], cats['parentid']))

def depth(cat_id: int, max_d: int = 20) -> int:
    d = 0
    while pd.notna(parent_map.get(cat_id)) and d < max_d:
        cat_id = int(parent_map[cat_id])
        d += 1
    return d

cats['depth'] = cats['categoryid'].apply(depth)

print('Distribuição de profundidade da árvore:')
print(cats['depth'].value_counts().sort_index().to_frame('categorias'))

fig, ax = plt.subplots(figsize=(7, 4))
cats['depth'].value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Profundidade da Árvore de Categorias')
ax.set_xlabel('Nível de Profundidade')
ax.set_ylabel('Número de Categorias')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()

- Hierarquia rasa (maioria em 2–3 níveis) — favorece agrupamentos de itens por categoria-pai
- Categorias raiz são pontos de entrada para navegação — agrupamento em nível 1 pode servir como feature categórica
- A profundidade máxima pequena indica taxonomia simples, diferente de marketplaces como Amazon

# Esparsidade da Matriz Usuário × Item

In [ ]:
n_users = events['visitorid'].nunique()
n_items = events['itemid'].nunique()
n_interacoes = len(events)
sparsity = 1 - n_interacoes / (n_users * n_items)

print(f'Usuários       : {n_users:>10,}')
print(f'Itens          : {n_items:>10,}')
print(f'Interações     : {n_interacoes:>10,}')
print(f'Tamanho matriz : {n_users * n_items:>10,}')
print(f'Esparsidade    : {sparsity:>10.4%}')

- Esparsidade acima de **99,99%** — extremamente típico em sistemas de recomendação
- A maioria absoluta dos pares (usuário, item) nunca foi observada
- Isso inviabiliza abordagens baseadas em memória pura (user-based CF) para escala real
- Modelos de **fatoração de matrizes** (MLP com embeddings) são mais adequados para este nível de esparsidade

# Análise de Associação

In [ ]:
# Métricas de engajamento por item
item_metrics = events.groupby('itemid')['event'].value_counts().unstack(fill_value=0)
for col in ['view', 'addtocart', 'transaction']:
    if col not in item_metrics.columns:
        item_metrics[col] = 0

item_metrics['total'] = item_metrics.sum(axis=1)
item_metrics['conv_rate'] = (item_metrics['transaction'] / item_metrics['view'].replace(0, np.nan)).fillna(0)
item_metrics.head()

In [ ]:
corr_cols = ['view', 'addtocart', 'transaction', 'total']

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    item_metrics[corr_cols].corr(),
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    vmin=-1,
    vmax=1,
    ax=ax,
)
ax.set_title('Correlação entre Métricas de Engajamento por Item')
plt.tight_layout()

- `view` e `addtocart` apresentam alta correlação — itens muito vistos tendem a ser adicionados ao carrinho
- `transaction` tem correlação menor com `view` do que com `addtocart` — indicando que a intenção de compra não é proporcional ao tráfego
- `total` é quase idêntico a `view` por ser dominado pelo volume de visualizações
- `conv_rate` (taxa view→transaction) é a métrica que captura qualidade, não volume

In [ ]:
# Amostra de itens para pairplot (evitar OOM)
sample_items = item_metrics[corr_cols].sample(min(5000, len(item_metrics)), random_state=42)

# Aplicar log1p para reduzir efeito de outliers
sample_log = np.log1p(sample_items)

sns.pairplot(sample_log, kind='scatter', plot_kws={'alpha': 0.2, 's': 10})
plt.suptitle('Pairplot — Métricas de Itens (log1p)', y=1.01, fontsize=12)
plt.show()

- Relação view–addtocart é quase linear em escala log — padrão consistente de engajamento
- `transaction` mostra maior dispersão — conversão é influenciada por fatores externos (preço, disponibilidade)
- Distribuições em escala log são mais próximas de normais — transformação log1p é adequada para features numéricas

# Árvore de Decisão — Drivers de Popularidade

## Preparando os Dados

In [ ]:
# Adiciona categoria ao item_metrics via item_properties
cat_lookup = (
    props[props['property'] == 'categoryid']
    .drop_duplicates('itemid')
    .set_index('itemid')['value']
    .rename('categoryid')
)

df_model = item_metrics[['view', 'addtocart', 'transaction']].copy()
df_model = df_model.join(cat_lookup)
df_model['has_category'] = df_model['categoryid'].notna().astype(int)
df_model['is_popular'] = (df_model['view'] >= df_model['view'].quantile(0.75)).astype(int)

print('Distribuição do target is_popular:')
print(df_model['is_popular'].value_counts(normalize=True).round(3))

In [ ]:
features = ['view', 'addtocart', 'transaction', 'has_category']
X = df_model[features].fillna(0)
y = df_model['is_popular']

tree_clf = DecisionTreeClassifier(min_samples_leaf=200, random_state=42)
tree_clf.fit(X, y)

In [ ]:
fig, ax = plt.subplots(figsize=(18, 7))
plot_tree(
    tree_clf,
    feature_names=features,
    class_names=['Não Popular', 'Popular'],
    filled=True,
    fontsize=11,
    ax=ax,
)
ax.set_title('Árvore de Decisão — Drivers de Popularidade de Itens', fontsize=14)
plt.tight_layout()

In [ ]:
importances = pd.Series(tree_clf.feature_importances_, index=features).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
importances.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Feature Importance — Popularidade do Item')
ax.set_xlabel('Importância Relativa')
ax.grid(True, axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()

print(importances.sort_values(ascending=False).to_frame('importance').round(4))

- `view` domina a importância como esperado — volume de visualizações é o principal determinante de popularidade
- `addtocart` contribui de forma complementar — captura intenção além do mero tráfego
- `transaction` tem peso menor individualmente, mas é o sinal de maior qualidade para aprendizado
- `has_category` impacto moderado — itens sem categoria tendem a ter menor visibilidade no catálogo

# Sugestões de Feature Engineering

Com base na análise exploratória, as seguintes features são candidatas prioritárias para o modelo:

### 1. Score de Engajamento Ponderado
Combina tipos de eventos com pesos refletindo intenção de compra:
```
engagement_score = views × 1 + addtocarts × 3 + transactions × 5
```

### 2. Nível de Atividade do Usuário
Classificação do usuário por volume de interações:
- `cold` : 1–4 interações
- `casual`: 5–20 interações  
- `active`: > 20 interações

### 3. Tier de Popularidade do Item
Classificação ordinal por percentil de visualizações:
- `long_tail`: abaixo do P50
- `mid_tier` : P50–P90
- `top_tier` : acima do P90

### 4. Features Temporais
- Hora do dia da interação (ciclo diário)
- Dia da semana (ciclo semanal)
- Dias desde a primeira interação do usuário (tenure)

### 5. Categoria do Item
- ID de categoria como variável categórica (embedding no modelo)
- Categoria raiz (nível 1 da hierarquia) como feature de agrupamento

### 6. Taxa de Conversão por Item
- `item_conv_rate = transactions / views` — proxy de qualidade do item
- Items com alta conv_rate mas baixo volume são candidatos a recomendação estratégica

# Principais Conclusões

- O dataset cobre **~4,5 meses** de comportamento real de e-commerce com padrões temporais consistentes (pico diurno, menor atividade nos fins de semana)

- A distribuição de interações tanto de usuários quanto de itens segue **power-law**, exigindo tratamento específico para itens e usuários frios

- O funil de conversão view → addtocart (~4%) → transaction é claro e diferencia bem os níveis de intenção de compra — base para weighting de feedback implícito

- A matriz usuário × item tem esparsidade acima de **99,99%**, tornando abordagens baseadas em fatoração de matrizes (embeddings) mais adequadas que métodos baseados em memória

- A propriedade `categoryid` é o único metadado estruturado confiável para content-based features

- `view` é o driver principal de popularidade, mas `addtocart` e `transaction` adicionam sinal de qualidade — ponderação dos eventos é essencial

# Próximos Passos

- Executar pipeline de pré-processamento (`dvc repro`) com filtragem de usuários frios e encoding de IDs
- Construir features de engajamento ponderado e tier de popularidade no estágio de feature engineering
- Implementar e avaliar baseline de popularidade (`PopularityRecommender`)
- Treinar modelo MLP com embeddings de usuário e item
- Avaliar com Recall@10 e NDCG@10 em split cronológico